In [ ]:
# Install dependencies
import sys, subprocess
pkg_list = ["neo4j", "openai", "numpy"]
for pkg in pkg_list:
    try:
        __import__(pkg)
        print(f"{pkg} already installed")
    except Exception:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
print("Dependencies ready")

In [ ]:
# Prepare connection info
import os
from getpass import getpass

NEO4J_URI = "neo4j://neo4j:7687"
NEO4J_USER = os.getenv("KNOWLEDGE_DB_USER")
NEO4J_PASSWORD = os.getenv("KNOWLEDGE_DB_PASSWORD")
if not NEO4J_PASSWORD:
    NEO4J_PASSWORD = getpass("Enter KNOWLEDGE_DB_PASSWORD (or set env KNOWLEDGE_DB_PASSWORD): ")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Enter OPENAI_API_KEY (or set OPENAI_API_KEY env): ")

print("Using NEO4J_URI=", NEO4J_URI)
print("Using KNOWLEDGE_DB_USER=", NEO4J_USER)

In [ ]:
# Connection test
from neo4j import GraphDatabase
try:
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        res = session.execute_read(lambda tx: tx.run("RETURN 1 AS v").single().get("v"))
    driver.close()
    print("Neo4j connection OK (test query returned):", res)
except Exception as e:
    print("Neo4j connection failed:", e)
    raise

# Function Definitions

Define all helper functions and phase functions.

In [ ]:
# Import functions
import json, re
from datetime import datetime
from pathlib import Path
from neo4j import GraphDatabase
from openai import OpenAI
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np

# Set OpenAI key
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# Config
DATA_DIR = "/workspace/outputs_sample/triples"
LOG_DIR = "/workspace/outputs_sample/clustering_logs"  # Log output path

# Similarity params
SIM_THRESHOLD = 0.7

# --- Helper Functions (Embedding, Cosine) ---
def get_embedding(openai_client, text: str, model: str = "text-embedding-3-small") -> list[float]:
    if not text or not isinstance(text, str):
        return []
    try:
        text = text.replace("\n", " ")
        resp = openai_client.embeddings.create(input=[text], model=model)
        return resp.data[0].embedding
    except Exception as e:
        print(f"⚠️ Embedding error: {e}")
        return []

def safe_rel_type(rt: str) -> str:
    if not rt:
        return "REL"
    return re.sub(r'\W+', '_', rt)

def _cosine(a: list[float], b: list[float]) -> float:
    a_arr = np.array(a, dtype=float)
    b_arr = np.array(b, dtype=float)
    if a_arr.size == 0 or b_arr.size == 0:
        return 0.0
    denom = (np.linalg.norm(a_arr) * np.linalg.norm(b_arr))
    if denom == 0:
        return 0.0
    return float(np.dot(a_arr, b_arr) / denom)

def get_weighted_embedding(openai_client, title: str, description: str, 
                           title_weight: float = 0.7, desc_weight: float = 0.3,
                           model: str = "text-embedding-3-small") -> list[float]:
    """
    Create a weighted vector from title and description
    Default: title 70%, description 30%
    """
    title = title or ""
    description = description or ""
    
    if not title and not description:
        return []
    
    try:
        # Get title vector
        title_vec = []
        if title:
            title_vec = get_embedding(openai_client, title, model)
        
        # Get description vector
        desc_vec = []
        if description:
            desc_vec = get_embedding(openai_client, description, model)
        
        # Apply weighting when both are available
        if title_vec and desc_vec:
            title_arr = np.array(title_vec, dtype=float)
            desc_arr = np.array(desc_vec, dtype=float)
            weighted_vec = (title_weight * title_arr + desc_weight * desc_arr)
            # Normalize
            norm = np.linalg.norm(weighted_vec)
            if norm > 0:
                weighted_vec = weighted_vec / norm
            return weighted_vec.tolist()
        elif title_vec:
            return title_vec
        elif desc_vec:
            return desc_vec
        else:
            return []
    except Exception as e:
        print(f"⚠️ Weighted embedding error: {e}")
        return []

print("✅ Helper functions defined")

In [ ]:
def import_file(tx, call_id: str, data: dict):
    nodes = data.get("Nodes", {})
    
    # 1. Create Nodes
    for label, items in nodes.items():
        if label == "Topic": continue
        safe_label = re.sub(r'\W+', '_', label)
        for item in items:
            node_id = item.get("id")
            if not node_id: continue
            props = {k: v for k, v in item.items() if k != "id"}
            
            if label in ("Issue", "Solution"):
                # Get title and description
                title = item.get("title", "") or item.get("name", "")
                desc = item.get("description", "") or item.get("text", "")
                
                if title or desc:
                    try:
                        # Create a weighted vector with title:description = 7:3
                        props["text_vector"] = get_weighted_embedding(
                            openai_client, title, desc, 
                            title_weight=0.7, desc_weight=0.3
                        )
                    except Exception as e:
                        print(f"    ⚠️ Embedding failed for {call_id}/{node_id}: {e}")
            
            query = (
                f"MERGE (n:{safe_label} {{id: $id, CALL_ID: $call_id}}) "
                "SET n += $props "
            )
            tx.run(query, id=node_id, call_id=call_id, props=props)

    # 2. Create Relations
    relations = data.get("Relations", [])
    for rel in relations:
        src = rel.get("source")
        tgt = rel.get("target")
        rtype = safe_rel_type(rel.get("type", "REL"))
        if not src or not tgt: continue
        query = (
            "MATCH (s {id:$src, CALL_ID:$call_id}), (t {id:$tgt, CALL_ID:$call_id}) "
            f"MERGE (s)-[r:`{rtype}`]->(t) "
            "SET r += $rel_props "
        )
        rel_props = {k: v for k, v in rel.items() if k not in ("source", "target")}
        tx.run(query, src=src, tgt=tgt, call_id=call_id, rel_props=rel_props)

    # 3. Rough Similarity Linking
    for label in ("Issue", "Solution"):
        safe_label = re.sub(r'\W+', '_', label)
        query_get_new = (
            f"MATCH (n:{safe_label}) "
            "WHERE n.CALL_ID = $call_id AND n.text_vector IS NOT NULL "
            "RETURN n.id AS id, n.CALL_ID AS call_id, n.text_vector AS vec"
        )
        new_records = list(tx.run(query_get_new, call_id=call_id))
        
        for rec in new_records:
            nid = rec["id"]
            nvec = rec["vec"] or []
            query_candidates = (
                f"MATCH (o:{safe_label}) "
                "WHERE NOT (o.id = $id AND o.CALL_ID = $call_id) AND o.text_vector IS NOT NULL "
                "RETURN o.id AS id, o.CALL_ID AS call_id, o.text_vector AS vec"
            )
            candidates = list(tx.run(query_candidates, id=nid, call_id=call_id))
            
            sims = []
            for c in candidates:
                cvec = c["vec"] or []
                score = _cosine(nvec, cvec)
                if score >= SIM_THRESHOLD:
                    sims.append((c["id"], c["call_id"], score))
            
            sims_sorted = sorted(sims, key=lambda x: x[2], reverse=True)
            for oid, ocall, score in sims_sorted:
                qlink = (
                    "MATCH (a {id:$a_id, CALL_ID:$a_call}), (b {id:$b_id, CALL_ID:$b_call}) "
                    "MERGE (a)-[r:IS_POSSIBLY_SIMILAR]-(b) "
                    "SET r.score = $score "
                )
                tx.run(qlink, a_id=nid, a_call=call_id, b_id=oid, b_call=ocall, score=score)

print("✅ Phase 2-1 functions defined")

In [ ]:
def find_connected_components(driver, type:str) -> list[list[dict]]:
    query = f"""
    MATCH (n:{type})-[r:IS_POSSIBLY_SIMILAR]-(m:{type})
    RETURN n.id AS id1, n.CALL_ID as call1, n.description as text1,
            m.id AS id2, m.CALL_ID as call2, m.description as text2
    """
    
    with driver.session() as session:
        records = list(session.run(query))
    
    adj = {}
    node_data = {}
    def get_key(nid, ncall): return f"{nid}___{ncall}"

    for r in records:
        k1 = get_key(r["id1"], r["call1"])
        k2 = get_key(r["id2"], r["call2"])
        node_data[k1] = {"id": r["id1"], "call_id": r["call1"], "text": r["text1"]}
        node_data[k2] = {"id": r["id2"], "call_id": r["call2"], "text": r["text2"]}
        if k1 not in adj: adj[k1] = []
        if k2 not in adj: adj[k2] = []
        adj[k1].append(k2)
        adj[k2].append(k1)
    
    visited = set()
    components = []
    for key in node_data:
        if key in visited: continue
        component_items = []
        stack = [key]
        visited.add(key)
        while stack:
            curr = stack.pop()
            component_items.append(node_data[curr])
            if curr in adj:
                for neighbor in adj[curr]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        stack.append(neighbor)
        components.append(component_items)
    return components

def classify_and_refine_cluster(driver, cluster_items, type:str) -> dict:
    """
    Process one cluster and return execution log info (dict).
    """
    # Data structure for logs
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "input_count": len(cluster_items),
        "input_items": [], # Before
        "llm_response": None,
        "created_groups": [], # After
        "created_relations": []
    }

    # Record input information
    for item in cluster_items:
        uid = f"{item['id']}___{item['call_id']}"
        log_entry["input_items"].append({
            "uid": uid,
            "text": item["text"]
        })

    if len(cluster_items) <= 1:
        log_entry["status"] = "skipped_single_node"
        return log_entry

    # 1. Build prompt
    items_for_prompt = [{"id": f"{i['id']}___{i['call_id']}", "text": i['text']} for i in cluster_items]
    items_json_str = json.dumps(items_for_prompt, ensure_ascii=False, indent=2)
    
    PROMPT_PATH = type == "Issue" and "/workspace/prompts_sample/clustering_issues.txt" or "/workspace/prompts_sample/clustering_solutions.txt"

    if not os.path.exists(PROMPT_PATH):
        print(f"❌ Prompt file missing: {PROMPT_PATH}")
        log_entry["error"] = "prompt_file_missing"
        return log_entry

    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        template = f.read()
    prompt = template.replace("{items_json}", items_json_str)

    # 2. LLM Call
    try:
        completion = openai_client.chat.completions.create(
            model="gpt-5",
            messages=[
                {"role": "system", "content": "You are a helpful assistant that outputs strictly JSON."},
                {"role": "user", "content": prompt}
            ],
        )
        response_text = completion.choices[0].message.content.strip()
        if response_text.startswith("```json"): response_text = response_text[7:]
        if response_text.endswith("```"): response_text = response_text[:-3]
        
        result = json.loads(response_text)
        log_entry["llm_response"] = result # Save raw LLM response
        
        # Get Nodes and Relations
        nodes = result.get("Nodes", [])
        relations = result.get("Relations", [])
        
    except Exception as e:
        print(f"    ❌ LLM Clustering Error: {e}")
        log_entry["error"] = str(e)
        return log_entry

    # 3. Update Neo4j
    all_uid_list = [x["id"] for x in items_for_prompt]
    
    with driver.session() as session:
        # Remove old links
        session.run("""
        UNWIND $uids AS uid
        WITH split(uid, "___") AS parts
        MATCH (n {id: parts[0], CALL_ID: parts[1]})-[r:IS_POSSIBLY_SIMILAR]-()
        DELETE r
        """, uids=all_uid_list)
        
        # Determine node type
        if type == "Issue":
            nodeType = "AbstractIssue"
        elif type == "Solution":
            nodeType = "AbstractSolution"
        
        # Create new groups and ID mapping
        abstract_node_map = {}  # Mapping from LLM-generated IDs to Neo4j elementId
        
        for node in nodes:
            node_id = node.get("id")
            label = node.get("label", "Unknown Group")
            description = node.get("description", "")
            member_uids = node.get("ids", [])
            
            # Create a new Abstract node even if ids is empty
            if not member_uids:
                # Create an abstract node with no members
                result = session.run(f"""
                CREATE (g:{nodeType} {{
                    llm_id: $llm_id,
                    name: $label,
                    description: $description,
                    validity: true,
                    source_call_ids: [],
                    created_at: timestamp()
                }})
                RETURN elementId(g) as element_id
                """, llm_id=node_id, label=label, description=description)
                
                record = result.single()
                if record:
                    element_id = record["element_id"]
                    abstract_node_map[node_id] = element_id
                    
                    # Record in log
                    log_entry["created_groups"].append({
                        "llm_id": node_id,
                        "label": label,
                        "element_id": element_id,
                        "members": [],
                        "source_call_ids": []
                    })
                    print(f"    ✨ Created Abstract Group (no members): '{label}' (ID: {node_id})")
            else:
                # Use the normal flow when members exist
                # Create Abstract node
                # Extract CALL_ID list from members
                result = session.run(f"""
                UNWIND $member_uids AS uid
                WITH split(uid, "___") AS parts
                MATCH (n {{id: parts[0], CALL_ID: parts[1]}})
                WITH collect(DISTINCT n.CALL_ID) as call_ids, collect(n) as members
                CREATE (g:{nodeType} {{
                    llm_id: $llm_id,
                    name: $label,
                    description: $description,
                    validity: true,
                    source_call_ids: call_ids,
                    created_at: timestamp()
                }})
                WITH g, members
                UNWIND members as n
                MERGE (n)-[:BELONGS_TO_GROUP]->(g)
                WITH g, collect(DISTINCT n.CALL_ID) as final_call_ids
                RETURN elementId(g) as element_id, final_call_ids as call_ids
                """, llm_id=node_id, label=label, description=description, member_uids=member_uids)
                
                record = result.single()
                if record:
                    element_id = record["element_id"]
                    call_ids = record["call_ids"]
                    abstract_node_map[node_id] = element_id
                    
                    # Record in log
                    log_entry["created_groups"].append({
                        "llm_id": node_id,
                        "label": label,
                        "element_id": element_id,
                        "members": member_uids,
                        "source_call_ids": call_ids
                    })
                    print(f"    ✨ Created Group: '{label}' (ID: {node_id}) with {len(member_uids)} members from {len(call_ids)} call(s).")
        
        # Create isSimilarTo relationships
        for rel in relations:
            source_id = rel.get("source")
            target_id = rel.get("target")
            rel_type = rel.get("type", "isSimilarTo")
            
            if source_id not in abstract_node_map or target_id not in abstract_node_map:
                print(f"    ⚠️ Skipping relation {source_id}->{target_id}: node not found")
                continue
            
            source_element_id = abstract_node_map[source_id]
            target_element_id = abstract_node_map[target_id]
            
            # Create relationship
            session.run(f"""
            MATCH (source:{nodeType}), (target:{nodeType})
            WHERE elementId(source) = $source_id AND elementId(target) = $target_id
            MERGE (source)-[r:{rel_type}]->(target)
            """, source_id=source_element_id, target_id=target_element_id)
            
            # Record in log
            log_entry["created_relations"].append({
                "source_llm_id": source_id,
                "target_llm_id": target_id,
                "type": rel_type,
                "source_element_id": source_element_id,
                "target_element_id": target_element_id
            })
            print(f"    🔗 Created relation: {source_id} -{rel_type}-> {target_id}")
            
    log_entry["status"] = "success"
    return log_entry

print("✅ Phase 2-2 functions defined")


In [ ]:
#   AND size(n.source_call_ids) >= 3
def get_eligible_abstract_nodes(driver) -> dict:
    """
    Get Abstract nodes where validity=true and source_call_ids has 3 or more items
    Returns: {"issues": [...], "solutions": [...]}
    """
    with driver.session() as session:
        # AbstractIssue
        issue_result = session.run("""
        MATCH (n:AbstractIssue)
        WHERE n.validity = true 
        
        RETURN elementId(n) as element_id, n.llm_id as llm_id, n.name as name, 
               n.source_call_ids as source_call_ids
        """)
        issues = [dict(record) for record in issue_result]
        
        # AbstractSolution
        solution_result = session.run("""
        MATCH (n:AbstractSolution)
        WHERE n.validity = true 

        RETURN elementId(n) as element_id, n.llm_id as llm_id, n.name as name,
               n.source_call_ids as source_call_ids
        """)
        solutions = [dict(record) for record in solution_result]
    
    return {"issues": issues, "solutions": solutions}


def get_connected_people(driver, abstract_node_type: str, element_id: str) -> list:
    """
    Get People connected from Abstract nodes via Issue/Solution and isHeldBy
    abstract_node_type: "AbstractIssue" or "AbstractSolution"
    """
    # AbstractIssue -> Issue -> People
    # AbstractSolution -> Solution -> People
    concrete_type = "Issue" if abstract_node_type == "AbstractIssue" else "Solution"
    
    with driver.session() as session:
        query = f"""
        MATCH (abstract:{abstract_node_type})
        WHERE elementId(abstract) = $element_id
        MATCH (abstract)<-[:BELONGS_TO_GROUP]-(concrete:{concrete_type})
        MATCH (concrete)-[:isHeldBy]->(people:People)
        RETURN DISTINCT people.id as id, people.CALL_ID as call_id,
               people.type as type, people.age as age, people.gender as gender,
               people.residence as residence, people.medicalHistory as medicalHistory,
               people.character as character, people.want as want,
               people.experience as experience, people.knowledgeLevel as knowledgeLevel,
               people.role as role
        """
        result = session.run(query, element_id=element_id)
        people_list = []
        for record in result:
            people_data = {k: record[k] for k in record.keys()}
            people_list.append(people_data)
    
    return people_list


def process_abstract_node_llm_only(driver_uri: str, user: str, password: str,
                                    abstract_type: str, element_id: str, 
                                    llm_id: str, name: str) -> dict:
    """
    Run only LLM processing for one Abstract node (split by type)
    Do not write to Neo4j; return results only
    Worker function for parallel processing
    """
    log_entry = {
        "timestamp": datetime.now().isoformat(),
        "abstract_type": abstract_type,
        "element_id": element_id,
        "llm_id": llm_id,
        "name": name,
        "people_count": 0,
        "status": "started",
        "grouped_people_data": []  # Store LLM results and metadata
    }
    
    # Use an independent driver in each thread
    thread_driver = GraphDatabase.driver(driver_uri, auth=(user, password))
    
    try:
        # 1. Retrieve People
        people_list = get_connected_people(thread_driver, abstract_type, element_id)
        log_entry["people_count"] = len(people_list)
        
        if len(people_list) == 0:
            log_entry["status"] = "no_people"
            return log_entry
        
        # 2. Group People by type
        people_by_type = {}
        for p in people_list:
            p_type = p.get("type", "Unknown")
            if p_type not in people_by_type:
                people_by_type[p_type] = []
            people_by_type[p_type].append(p)
        
        log_entry["people_types"] = {t: len(lst) for t, lst in people_by_type.items()}
        print(f"    📊 Found {len(people_by_type)} different People types: {list(people_by_type.keys())}")
        
        # 3. Run LLM processing per type (no Neo4j writes)
        for people_type, type_people_list in people_by_type.items():
            print(f"      🔄 Processing type '{people_type}' ({len(type_people_list)} people)...")
            
            # 3.1. Build prompt
            items_for_prompt = []
            for p in type_people_list:
                item = {
                    "id": p["id"],
                    "call_id": p["call_id"],
                }
                # Add each field only if not null
                for field in ["type", "age", "gender", "residence", "medicalHistory", 
                              "character", "want", "experience", "knowledgeLevel", "role"]:
                    if p.get(field):
                        item[field] = p[field]
                items_for_prompt.append(item)
            
            items_json_str = json.dumps(items_for_prompt, ensure_ascii=False, indent=2)
            
            PROMPT_PATH = "/workspace/prompts_sample/group_peoples.txt"
            if not os.path.exists(PROMPT_PATH):
                log_entry["status"] = "error"
                log_entry["error"] = "prompt_file_missing"
                return log_entry
            
            with open(PROMPT_PATH, "r", encoding="utf-8") as f:
                template = f.read()
            prompt = template.replace("{items_json}", items_json_str)
            
            # 3.2. LLM Call
            try:
                completion = openai_client.chat.completions.create(
                    model="gpt-5",
                    messages=[
                        {"role": "system", "content": "You are a helpful assistant that outputs strictly JSON."},
                        {"role": "user", "content": prompt}
                    ],
                )
                response_text = completion.choices[0].message.content.strip()
                if response_text.startswith("```json"): response_text = response_text[7:]
                if response_text.endswith("```"): response_text = response_text[:-3]
                
                result = json.loads(response_text)
                grouped_people_data = result.get("GroupedPeople", {})
                
                # Save LLM results (Neo4j writes happen later)
                log_entry["grouped_people_data"].append({
                    "people_type": people_type,
                    "people_count": len(type_people_list),
                    "llm_response": grouped_people_data,
                    "source_people": items_for_prompt
                })
                print(f"      ✅ LLM processed type '{people_type}'")
                
            except Exception as e:
                print(f"      ❌ LLM error for type '{people_type}': {e}")
                log_entry["grouped_people_data"].append({
                    "people_type": people_type,
                    "error": str(e)
                })
                continue
        
        log_entry["status"] = "success"
        print(f"    ✅ Completed LLM processing for {abstract_type} '{name}': {len(log_entry['grouped_people_data'])} types processed")
            
    except Exception as e:
        log_entry["status"] = "error"
        log_entry["error"] = str(e)
        print(f"    ❌ Error processing {abstract_type} '{name}': {e}")
    finally:
        thread_driver.close()
    
    return log_entry


def insert_grouped_people_to_neo4j(driver, log_entry: dict):
    """
    Insert LLM processing results into Neo4j (sequentially)
    """
    if log_entry.get("status") != "success":
        return
    
    abstract_type = log_entry["abstract_type"]
    element_id = log_entry["element_id"]
    name = log_entry["name"]
    
    created_count = 0
    
    with driver.session() as session:
        for gp_data in log_entry.get("grouped_people_data", []):
            if "error" in gp_data:
                continue
            
            people_type = gp_data["people_type"]
            llm_response = gp_data["llm_response"]
            
            # Generate unique ID (element_id + type)
            import hashlib
            id_hash = hashlib.md5(element_id.encode()).hexdigest()[:8]
            safe_type = people_type.replace(' ', '_').replace('/', '_')
            gp_id = f"GP_{id_hash}_{safe_type}"
            
            gp_type = llm_response.get("type", people_type)
            source_ids = llm_response.get("source_ids", [])
            
            # Extract CALL_ID list from source_ids
            source_call_ids = list(set([src["call_id"] for src in source_ids if "call_id" in src]))
            
            # Prepare properties
            props = {
                "gp_id": gp_id,
                "type": gp_type,
                "abstract_element_id": element_id,
                "abstract_name": name,
                "source_call_ids": source_call_ids,
            }
            for field in ["age", "residence", "medicalHistory", 
                          "character", "want", "experience", "knowledgeLevel", "role"]:
                if field in llm_response and llm_response[field]:
                    props[field] = llm_response[field]
            
            # Create GroupedPeople nodes and connect to Abstract nodes
            session.run("""
            CREATE (gp:GroupedPeople)
            SET gp += $props, gp.created_at = timestamp()
            
            WITH gp
            MATCH (abstract)
            WHERE elementId(abstract) = $element_id
            MERGE (abstract)-[:hasGroupedPeople]->(gp)
            """, props=props, element_id=element_id)
            
            # Connect original People to GroupedPeople
            if source_ids:
                for src in source_ids:
                    session.run("""
                    MATCH (p:People {id: $id, CALL_ID: $call_id})
                    MATCH (gp:GroupedPeople {gp_id: $gp_id})
                    MERGE (p)-[:belongsToGroupedPeople]->(gp)
                    """, id=src["id"], call_id=src["call_id"], gp_id=gp_id)
            
            created_count += 1
            print(f"      💾 Inserted GroupedPeople '{gp_id}' for type '{people_type}'")
    
    print(f"    💾 Inserted {created_count} GroupedPeople nodes for {abstract_type} '{name}'")

print("✅ Phase 2-3 functions defined")

In [ ]:
def find_islands_with_grouped_people(driver) -> list[dict]:
    """
    Identify islands containing at least one GroupedPeople, AbstractIssue, and AbstractSolution
    Target nodes:
    - source_call_ids >= 3 and has GroupedPeople
    - source_call_ids = 0 and no GroupedPeople
    Returns: a list containing information for each island
    """
    with driver.session() as session:
        # 1. Retrieve target Abstract nodes (validity=true)
        query = """
        MATCH (abstract)
        WHERE (abstract:AbstractIssue OR abstract:AbstractSolution)
          AND abstract.validity = true
        RETURN DISTINCT elementId(abstract) as element_id, 
               labels(abstract)[0] as label,
               EXISTS { MATCH (abstract)-[:hasGroupedPeople]->(:GroupedPeople) } as has_grouped_people,
               size(abstract.source_call_ids) as source_call_ids_size
        """
        result = session.run(query)
        eligible_nodes = []
        nodes_with_gp = set()
        
        for r in result:
            element_id = r["element_id"]
            label = r["label"]
            has_gp = r["has_grouped_people"]
            call_size = r["source_call_ids_size"]
            
            # Target node conditions:
            # (source_call_ids >= 3 and has GroupedPeople) OR (source_call_ids = 0 and no GroupedPeople)
            # is_eligible = (call_size >= 3 and has_gp) or (call_size == 0 and not has_gp)
            is_eligible = True
            
            if is_eligible:
                eligible_nodes.append({"element_id": element_id, "label": label})
                # Record nodes with GroupedPeople (for island filtering)
                if has_gp:
                    nodes_with_gp.add(element_id)
        
        print(f"Found {len(eligible_nodes)} eligible Abstract nodes:")
        print(f"  - {len(nodes_with_gp)} nodes with GroupedPeople (source_call_ids >= 3)")
        print(f"  - {len(eligible_nodes) - len(nodes_with_gp)} nodes without GroupedPeople (source_call_ids = 0)")
        
        if not eligible_nodes:
            return []
        
        # 2. Build graph structure (undirected graph)
        adj = {}
        node_labels = {}
        
        for node in eligible_nodes:
            eid = node["element_id"]
            adj[eid] = []
            node_labels[eid] = node["label"]
        
        # Get relationships between Abstract nodes (treated as undirected)
        rel_query = """
        MATCH (a)-[r]-(b)
        WHERE (a:AbstractIssue OR a:AbstractSolution)
          AND (b:AbstractIssue OR b:AbstractSolution)
          AND a.validity = true
          AND b.validity = true
          AND type(r) IN ['isResolvedBy', 'isGeneratedBy', 'isPartOf']
        RETURN DISTINCT elementId(a) as a_id, elementId(b) as b_id, type(r) as rel_type
        """
        rel_result = session.run(rel_query)
        
        for rel in rel_result:
            a_id = rel["a_id"]
            b_id = rel["b_id"]
            # Connect only if both nodes are in eligible_nodes
            if a_id in adj and b_id in adj:
                if b_id not in adj[a_id]:
                    adj[a_id].append(b_id)
                if a_id not in adj[b_id]:
                    adj[b_id].append(a_id)
        
        # 3. Extract connected components (islands)
        visited = set()
        islands = []
        
        for start_node in adj.keys():
            if start_node in visited:
                continue
            
            # Build islands with BFS
            island_nodes = []
            queue = [start_node]
            visited.add(start_node)
            
            while queue:
                current = queue.pop(0)
                island_nodes.append(current)
                
                for neighbor in adj[current]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append(neighbor)
            
            # Check whether island contains nodes with GroupedPeople (source_call_ids >= 3 or = 0)
            has_gp_in_island = any(node in nodes_with_gp for node in island_nodes)
            
            if not has_gp_in_island:
                continue  # Skip islands with no nodes that have GroupedPeople
            
            # Check island statistics
            issue_count = sum(1 for n in island_nodes if node_labels[n] == "AbstractIssue")
            solution_count = sum(1 for n in island_nodes if node_labels[n] == "AbstractSolution")
            
            # Keep only islands with at least one Issue and one Solution
            if issue_count >= 1 and solution_count >= 1:
                islands.append({
                    "nodes": island_nodes,
                    "issue_count": issue_count,
                    "solution_count": solution_count
                })
        
        print(f"Found {len(islands)} islands with both AbstractIssue and AbstractSolution (and GroupedPeople)")
        
        return islands


def extract_island_data(driver, island_nodes: list[str]) -> dict:
    """
    Extract all information contained in one island
    Returns: {
        "abstract_issues": [...],
        "abstract_solutions": [...],
        "relations": [...]
    }
    """
    island_data = {
        "abstract_issues": [],
        "abstract_solutions": [],
        "relations": []
    }
    
    with driver.session() as session:
        # 1. Retrieve AbstractIssue node info (including GroupedPeople)
        for node_id in island_nodes:
            # Retrieve basic node information
            node_query = """
            MATCH (n)
            WHERE elementId(n) = $node_id
            RETURN labels(n)[0] as label, properties(n) as props, elementId(n) as element_id
            """
            node_result = session.run(node_query, node_id=node_id)
            node_record = node_result.single()
            
            if not node_record:
                continue
            
            label = node_record["label"]
            props = dict(node_record["props"])
            element_id = node_record["element_id"]
            
            # Retrieve GroupedPeople
            gp_query = """
            MATCH (abstract)-[:hasGroupedPeople]->(gp:GroupedPeople)
            WHERE elementId(abstract) = $element_id
            RETURN properties(gp) as gp_props
            """
            gp_result = session.run(gp_query, element_id=element_id)
            grouped_people_list = [dict(r["gp_props"]) for r in gp_result]
            
            node_info = {
                "element_id": element_id,
                "properties": props,
                "grouped_people": grouped_people_list
            }
            
            if label == "AbstractIssue":
                island_data["abstract_issues"].append(node_info)
            elif label == "AbstractSolution":
                island_data["abstract_solutions"].append(node_info)
        
        # 2. Retrieve relationship info (excluding GroupedPeople-related)
        relations_query = """
        UNWIND $node_ids as node_id
        MATCH (a)-[r]->(b)
        WHERE elementId(a) = node_id
          AND elementId(b) IN $node_ids
          AND type(r) IN ['isResolvedBy', 'isGeneratedBy', 'isSimilarTo', 'isPartOf']
        RETURN elementId(a) as source_id, elementId(b) as target_id, 
               type(r) as rel_type, properties(r) as rel_props
        """
        rel_result = session.run(relations_query, node_ids=island_nodes)
        
        for rel in rel_result:
            island_data["relations"].append({
                "source": rel["source_id"],
                "target": rel["target_id"],
                "type": rel["rel_type"],
                "properties": dict(rel["rel_props"]) if rel["rel_props"] else {}
            })
    
    return island_data

print("✅ Phase 3-1 functions defined")

In [ ]:
def create_personas_for_island(island_data: dict) -> dict:
    """
    Create personas from one island's data
    Args:
        island_data: island data extracted in Phase 5
    Returns:
        Persona information generated by the LLM
    """
    # 1. Format data for prompt
    island_summary = {
        "island_id": island_data.get("island_id"),
        "abstract_issues": [],
        "abstract_solutions": [],
        "relations": island_data.get("relations", [])
    }
    
    # Format AbstractIssue information
    for ai in island_data.get("abstract_issues", []):
        issue_info = {
            "element_id": ai["element_id"],
            "name": ai["properties"].get("name", ""),
            "description": ai["properties"].get("description", ""),
            "source_call_ids": ai["properties"].get("source_call_ids", []),
            "grouped_people": []
        }
        
        # Add GroupedPeople information
        for gp in ai.get("grouped_people", []):
            gp_info = {
                "gp_id": gp.get("gp_id", ""),
                "type": gp.get("type", ""),
                "age": gp.get("age"),
                "residence": gp.get("residence"),
                "character": gp.get("character"),
                "want": gp.get("want"),
                "experience": gp.get("experience"),
                "knowledgeLevel": gp.get("knowledgeLevel"),
                "role": gp.get("role"),
                "source_call_ids": gp.get("source_call_ids", [])
            }
            # Exclude None values
            gp_info = {k: v for k, v in gp_info.items() if v is not None}
            issue_info["grouped_people"].append(gp_info)
        
        island_summary["abstract_issues"].append(issue_info)
    
    # Format AbstractSolution information
    for asol in island_data.get("abstract_solutions", []):
        solution_info = {
            "element_id": asol["element_id"],
            "name": asol["properties"].get("name", ""),
            "description": asol["properties"].get("description", ""),
            "source_call_ids": asol["properties"].get("source_call_ids", []),
            "grouped_people": []
        }
        
        # Add GroupedPeople information
        for gp in asol.get("grouped_people", []):
            gp_info = {
                "gp_id": gp.get("gp_id", ""),
                "type": gp.get("type", ""),
                "age": gp.get("age"),
                "residence": gp.get("residence"),
                "character": gp.get("character"),
                "want": gp.get("want"),
                "experience": gp.get("experience"),
                "knowledgeLevel": gp.get("knowledgeLevel"),
                "role": gp.get("role"),
                "source_call_ids": gp.get("source_call_ids", [])
            }
            # Exclude None values
            gp_info = {k: v for k, v in gp_info.items() if v is not None}
            solution_info["grouped_people"].append(gp_info)
        
        island_summary["abstract_solutions"].append(solution_info)
    
    # 2. Serialize JSON to string
    island_json_str = json.dumps(island_summary, ensure_ascii=False, indent=2)
    
    # 3. Load prompt
    PROMPT_PATH = "/workspace/prompts_sample/create_personas.txt"
    if not os.path.exists(PROMPT_PATH):
        return {
            "status": "error",
            "error": "prompt_file_missing",
            "island_id": island_data.get("island_id")
        }
    
    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        template = f.read()
    
    prompt = template.replace("{island_data}", island_json_str)
    
    # 4. LLM Call
    try:
        completion = openai_client.chat.completions.create(
            model="gpt-5",
            messages=[
                {"role": "system", "content": "You are an expert in insurance industry business analysis and thought simulation. Output strictly valid JSON."},
                {"role": "user", "content": prompt}
            ],

        )
        response_text = completion.choices[0].message.content.strip()
        
        # Remove code block markers
        if response_text.startswith("```json"):
            response_text = response_text[7:]
        if response_text.startswith("```"):
            response_text = response_text[3:]
        if response_text.endswith("```"):
            response_text = response_text[:-3]
        
        result = json.loads(response_text)
        result["status"] = "success"
        result["island_id"] = island_data.get("island_id")
        
        return result
        
    except Exception as e:
        print(f"❌ LLM Persona Creation Error for island {island_data.get('island_id')}: {e}")
        return {
            "status": "error",
            "error": str(e),
            "island_id": island_data.get("island_id")
        }


def process_personas_parallel(islands_data: list[dict], max_workers: int = 5) -> list[dict]:
    """
    Create personas for multiple islands in parallel
    Args:
        islands_data: all island data extracted in Phase 5
        max_workers: maximum number of parallel workers
    Returns:
        List of persona data for all islands
    """
    all_personas = []
    
    print(f"🚀 Creating personas for {len(islands_data)} islands in parallel (max_workers={max_workers})...")
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(create_personas_for_island, island): island["island_id"]
            for island in islands_data
        }
        
        completed = 0
        for future in as_completed(futures):
            island_id = futures[future]
            completed += 1
            try:
                persona_data = future.result()
                all_personas.append(persona_data)
                
                if persona_data.get("status") == "success":
                    scenario_pair_count = len(persona_data.get("scenario_pairs", []))
                    print(f"✅ Island {island_id}: Created {scenario_pair_count} scenario pairs ({completed}/{len(islands_data)})")
                else:
                    print(f"⚠️ Island {island_id}: {persona_data.get('error', 'Unknown error')} ({completed}/{len(islands_data)})")
                    
            except Exception as e:
                print(f"❌ Error processing island {island_id}: {e}")
                all_personas.append({
                    "status": "error",
                    "error": str(e),
                    "island_id": island_id
                })
    
    return all_personas

print("✅ Phase 3-2 functions defined")

In [ ]:
# Utility Functions

def clear_neo4j(driver):
    """Delete all nodes/relationships (use with caution)."""
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("✅ Cleared all nodes and relationships in Neo4j.")

def setup_neo4j_indices(driver):
    """Create basic indexes/constraints to speed lookups (id + CALL_ID)."""
    with driver.session() as session:
        # Basic composite indexes for quick MERGE/MATCH by id+CALL_ID
        session.run("CREATE INDEX IF NOT EXISTS FOR (n:Issue) ON (n.id, n.CALL_ID)")
        session.run("CREATE INDEX IF NOT EXISTS FOR (n:Solution) ON (n.id, n.CALL_ID)")
        session.run("CREATE INDEX IF NOT EXISTS FOR (n:People) ON (n.id, n.CALL_ID)")
        
        # Add an index on GroupedPeople.gp_id (faster lookup)
        session.run("CREATE INDEX IF NOT EXISTS FOR (n:GroupedPeople) ON (n.gp_id)")
        print("✅ Created index on GroupedPeople.gp_id")
    
    print("✅ Created basic indexes (Issue/Solution/People on id + CALL_ID).")

print("✅ Utility functions defined")

# Setup

Create Neo4j indexes and clear existing data.

In [ ]:
# Run setup
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
setup_neo4j_indices(driver)
clear_neo4j(driver)
driver.close()
print("✅ Setup completed")

# Phase 1: Extract Triples

In [ ]:
import os
import json
from pathlib import Path
from langchain_openai import ChatOpenAI
from utils.langchain import Langchain
from utils.file_reader import FileReader
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Config ---
dataset_dir = "/workspace/data_sample"
output_dir_1 = "/workspace/outputs_sample/triples"
prompt_path = "/workspace/prompts_sample/log_extraction.txt"

os.makedirs(output_dir_1, exist_ok=True)

with open(prompt_path, "r", encoding="utf-8") as f:
    PROMPT = f.read()

# ---  / helpers ---
llm = ChatOpenAI(
    model="gpt-5",
    api_key=OPENAI_API_KEY,
    max_tokens=None,
    timeout=None,
    max_retries=3,
)

langchain = Langchain(llm)
file_reader = FileReader()

def clean_response(content: str) -> str:
    if content.startswith("```"):
        content = content[3:]
    if content.startswith("json"):
        content = content[4:]
    if content.endswith("```"):
        content = content[:-3]
    return content.strip()

def process(fname: str) -> tuple[str, bool]:
    file_id = Path(fname).stem
    src_path = os.path.join(dataset_dir, fname)
    conversation = file_reader.read_file(src_path)
    
    step1_prompt = PROMPT.replace("$conversation", conversation)
    res1 = langchain.generate_prompt_only(step1_prompt)
    content1 = getattr(res1, "content", "") if res1 is not None else ""
    content1 = clean_response(content1)

    out1_path = os.path.join(output_dir_1, f"{file_id}.json")
    with open(out1_path, "w", encoding="utf-8") as f:
        f.write(content1)

    print(f"Wrote {out1_path}")
    return file_id, True

# --- Process files ---
txt_files = [f for f in os.listdir(dataset_dir) if f.endswith(".txt")]
txt_files = sorted(txt_files)

# Run Phase 1
print("🚀 PHASE 1: Extracting triples...")

with ThreadPoolExecutor() as executor:
    futures = {executor.submit(process, fname): fname for fname in txt_files}
    for future in as_completed(futures):
        fname = futures[future]
        try:
            future.result()
        except Exception as e:
            print(f"⚠️ Error processing {fname}: {e}")

print("✅ Phase 1 completed")

# Phase 2-1: Create rough connections

In [ ]:
# Run Phase 2-1
print("🚀 PHASE 2-1: Importing files and creating rough connections...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".json")])
print(f"Found {len(files)} files to process")

if not files:
    print("No JSON files found.")
else:
    with driver.session() as session:
        for idx, fname in enumerate(files, 1):
            path = Path(DATA_DIR) / fname
            call_id = path.stem
            print(f"[{idx}/{len(files)}] Processing {fname}...", end=" ", flush=True)
            try:
                with open(path, "r", encoding="utf-8") as fh:
                    data = json.load(fh)
                session.execute_write(import_file, call_id, data)
                print("✓ done")
            except Exception as e:
                print(f"✗ error: {e}")

driver.close()
print("✅ Phase 2-1 completed")

# Phase 2-2: Creation of AbstractNodes

In [ ]:
# Run Phase 2-2
print("🔍 PHASE 2-2: Extracting islands and refining with LLM...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# List for collecting logs
all_cluster_logs = []

# Helper function for parallel processing
def process_cluster_wrapper(args):
    """Wrapper function for parallel processing"""
    driver_uri, user, password, cluster, cluster_type, cluster_index = args
    # Use an independent driver in each thread
    thread_driver = GraphDatabase.driver(driver_uri, auth=(user, password))
    try:
        log_data = classify_and_refine_cluster(thread_driver, cluster, cluster_type)
        if log_data:
            log_data["cluster_index"] = cluster_index
            log_data["cluster_type"] = cluster_type
        return log_data
    finally:
        thread_driver.close()

# Issue clustering
issue_clusters = find_connected_components(driver, "Issue")
print(f"Found {len(issue_clusters)} connected issue clusters (islands) to analyze.")

if issue_clusters:
    print(f"Processing Issue clusters in parallel (max_workers={min(len(issue_clusters), 10)})...")
    issue_tasks = [
        (NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, cluster, "Issue", i)
        for i, cluster in enumerate(issue_clusters)
    ]
    
    with ThreadPoolExecutor(max_workers=min(len(issue_clusters), 10)) as executor:
        futures = {executor.submit(process_cluster_wrapper, task): task for task in issue_tasks}
        
        for future in as_completed(futures):
            task = futures[future]
            cluster_index = task[5]
            try:
                log_data = future.result()
                if log_data:
                    all_cluster_logs.append(log_data)
                    print(f"✅ Completed Issue Cluster {cluster_index + 1}/{len(issue_clusters)}")
            except Exception as e:
                print(f"❌ Error processing Issue Cluster {cluster_index + 1}: {e}")

# Solution clustering
solution_clusters = find_connected_components(driver, "Solution")
print(f"\nFound {len(solution_clusters)} connected solution clusters (islands) to analyze.")

if solution_clusters:
    print(f"Processing Solution clusters in parallel (max_workers={min(len(solution_clusters), 10)})...")
    solution_tasks = [
        (NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, cluster, "Solution", i)
        for i, cluster in enumerate(solution_clusters)
    ]
    
    with ThreadPoolExecutor(max_workers=min(len(solution_clusters), 10)) as executor:
        futures = {executor.submit(process_cluster_wrapper, task): task for task in solution_tasks}
        
        for future in as_completed(futures):
            task = futures[future]
            cluster_index = task[5]
            try:
                log_data = future.result()
                if log_data:
                    all_cluster_logs.append(log_data)
                    print(f"✅ Completed Solution Cluster {cluster_index + 1}/{len(solution_clusters)}")
            except Exception as e:
                print(f"❌ Error processing Solution Cluster {cluster_index + 1}: {e}")

# Create relationships between Abstract nodes
print("\n🔗 Creating relationships between AbstractIssue and AbstractSolution...")
with driver.session() as session:
    # isResolvedBy relationships
    result = session.run("""
    MATCH (ai:AbstractIssue)<-[:BELONGS_TO_GROUP]-(issue:Issue)
    MATCH (issue)-[:isResolvedBy]->(solution:Solution)
    MATCH (solution)-[:BELONGS_TO_GROUP]->(as:AbstractSolution)
    WITH ai, as, count(*) as connection_count
    MERGE (ai)-[r:isResolvedBy]->(as)
    SET r.connection_count = connection_count
    WITH count(r) as rel_count
    RETURN rel_count
    """)
    record = result.single()
    rel_count = record["rel_count"] if record else 0
    print(f"✅ Created {rel_count} isResolvedBy relationships between Abstract nodes")

    # isGeneratedBy relationships (AbstractIssue <- AbstractSolution)
    result = session.run("""
    MATCH (ai:AbstractIssue)<-[:BELONGS_TO_GROUP]-(issue:Issue)
    MATCH (issue)-[:isGeneratedBy]->(solution:Solution)
    MATCH (solution)-[:BELONGS_TO_GROUP]->(as:AbstractSolution)
    WITH ai, as, count(*) as connection_count
    MERGE (ai)-[r:isGeneratedBy]->(as)
    SET r.connection_count = connection_count
    WITH count(r) as rel_count
    RETURN rel_count
    """)
    record = result.single()
    rel_count = record["rel_count"] if record else 0
    print(f"✅ Created {rel_count} isGeneratedBy relationships between Abstract nodes")

    # isGeneratedBy relationships (AbstractIssue <- AbstractIssue)
    result = session.run("""
    MATCH (ai1:AbstractIssue)<-[:BELONGS_TO_GROUP]-(issue1:Issue)
    MATCH (issue1)-[:isGeneratedBy]->(issue2:Issue)
    MATCH (issue2)-[:BELONGS_TO_GROUP]->(ai2:AbstractIssue)
    WHERE ai1 <> ai2
    WITH ai1, ai2, count(*) as connection_count
    MERGE (ai1)-[r:isGeneratedBy]->(ai2)
    SET r.connection_count = connection_count
    WITH count(r) as rel_count
    RETURN rel_count
    """)
    record = result.single()
    rel_count = record["rel_count"] if record else 0
    print(f"✅ Created {rel_count} isGeneratedBy relationships between Abstract nodes")

driver.close()

# Save Phase 2-2 logs
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"clustering_log_{timestamp}.json"
log_path = os.path.join(LOG_DIR, log_filename)

try:
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(all_cluster_logs, f, ensure_ascii=False, indent=2)
    print(f"📝 Clustering log saved to: {log_path}")
except Exception as e:
    print(f"❌ Failed to save clustering log: {e}")

print("✅ Phase 2-2 completed")


# Phase 2-3: Creation of GroupedPeople

In [ ]:
# Run Phase 2-3
print("👥 PHASE 2-3: Creating GroupedPeople nodes...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# List for collecting logs
all_grouped_people_logs = []

# 1. Retrieve target Abstract nodes
print("🔍 Finding eligible Abstract nodes (validity=true, source_call_ids >= 3)...")
eligible_nodes = get_eligible_abstract_nodes(driver)

issue_nodes = eligible_nodes["issues"]
solution_nodes = eligible_nodes["solutions"]

print(f"Found {len(issue_nodes)} AbstractIssue nodes")
print(f"Found {len(solution_nodes)} AbstractSolution nodes")
print(f"Total: {len(issue_nodes) + len(solution_nodes)} nodes to process")

# 2. Run LLM processing in parallel (no Neo4j writes)
all_tasks = []

# AbstractIssue tasks
for node in issue_nodes:
    all_tasks.append({
        "abstract_type": "AbstractIssue",
        "element_id": node["element_id"],
        "llm_id": node["llm_id"],
        "name": node["name"]
    })

# AbstractSolution tasks
for node in solution_nodes:
    all_tasks.append({
        "abstract_type": "AbstractSolution",
        "element_id": node["element_id"],
        "llm_id": node["llm_id"],
        "name": node["name"]
    })

if all_tasks:
    print(f"\n🚀 Step 1: Running LLM processing in parallel (max_workers={min(len(all_tasks), 10)})...")
    
    with ThreadPoolExecutor(max_workers=min(len(all_tasks), 10)) as executor:
        futures = {
            executor.submit(
                process_abstract_node_llm_only,
                NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD,
                task["abstract_type"], task["element_id"], 
                task["llm_id"], task["name"]
            ): task
            for task in all_tasks
        }
        
        completed = 0
        for future in as_completed(futures):
            task = futures[future]
            completed += 1
            try:
                log_data = future.result()
                if log_data:
                    all_grouped_people_logs.append(log_data)
                print(f"LLM Progress: {completed}/{len(all_tasks)} completed")
            except Exception as e:
                print(f"❌ Error processing {task['abstract_type']} '{task['name']}': {e}")
    
    print(f"\n✅ Step 1 completed: {len(all_grouped_people_logs)} nodes processed by LLM")
    
    # 3. Insert into Neo4j sequentially (not parallel)
    print(f"\n💾 Step 2: Inserting GroupedPeople nodes to Neo4j sequentially...")
    success_logs = [log for log in all_grouped_people_logs if log.get("status") == "success"]
    
    for idx, log_entry in enumerate(success_logs, 1):
        try:
            insert_grouped_people_to_neo4j(driver, log_entry)
            print(f"Insert Progress: {idx}/{len(success_logs)} completed")
        except Exception as e:
            print(f"❌ Error inserting for {log_entry['abstract_type']} '{log_entry['name']}': {e}")
            log_entry["insert_error"] = str(e)
    
    print(f"\n✅ Step 2 completed: GroupedPeople nodes inserted")

driver.close()

# Save Phase 2-3 logs
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"grouped_people_log_{timestamp}.json"
log_path = os.path.join(LOG_DIR, log_filename)

try:
    with open(log_path, "w", encoding="utf-8") as f:
        json.dump(all_grouped_people_logs, f, ensure_ascii=False, indent=2)
    print(f"📝 GroupedPeople log saved to: {log_path}")
except Exception as e:
    print(f"❌ Failed to save GroupedPeople log: {e}")

# Show statistics
success_count = sum(1 for log in all_grouped_people_logs if log.get("status") == "success")
no_people_count = sum(1 for log in all_grouped_people_logs if log.get("status") == "no_people")
error_count = sum(1 for log in all_grouped_people_logs if log.get("status") == "error")

# Compute total number of GroupedPeople nodes
total_grouped_people_nodes = sum(
    len([gp for gp in log.get("grouped_people_data", []) if "error" not in gp])
    for log in all_grouped_people_logs 
    if log.get("status") == "success"
)

# Statistics by type
type_stats = {}
for log in all_grouped_people_logs:
    if log.get("status") == "success":
        for gp in log.get("grouped_people_data", []):
            if "error" in gp:
                continue
            gp_type = gp.get("people_type", "Unknown")
            if gp_type not in type_stats:
                type_stats[gp_type] = {"count": 0, "total_people": 0}
            type_stats[gp_type]["count"] += 1
            type_stats[gp_type]["total_people"] += gp.get("people_count", 0)

print(f"\n📊 Phase 2-3 Summary:")
print(f"  🎯 Abstract nodes processed: {len(all_grouped_people_logs)}")
print(f"    ✅ Successfully processed: {success_count} nodes")
print(f"    ⚠️  No people found: {no_people_count} nodes")
print(f"    ❌ Errors: {error_count} nodes")
print(f"\n  👥 GroupedPeople nodes created: {total_grouped_people_nodes}")
if type_stats:
    print(f"  📋 Breakdown by People type:")
    for ptype, stats in sorted(type_stats.items()):
        print(f"    • {ptype}: {stats['count']} GroupedPeople nodes ({stats['total_people']} people)")

print("✅ Phase 2-3 completed")

# Phase 3-1: Extraction of Connected Components

In [ ]:
# Run Phase 3-1: Extract and export islands
print("🏝️ PHASE 3-1: Extracting islands with GroupedPeople...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

# 1. Identify islands
print("🔍 Step 1: Identifying islands...")
islands = find_islands_with_grouped_people(driver)

if not islands:
    print("⚠️ No islands found with required conditions")
    driver.close()
else:
    print(f"\n📊 Found {len(islands)} qualifying islands:")
    for idx, island in enumerate(islands, 1):
        print(f"  Island {idx}: {island['issue_count']} AbstractIssues, {island['solution_count']} AbstractSolutions")
    
    # 2. Extract data from each island
    print(f"\n📦 Step 2: Extracting data from islands...")
    all_islands_data = []
    
    for idx, island in enumerate(islands, 1):
        print(f"\n  Processing Island {idx}...")
        island_data = extract_island_data(driver, island["nodes"])
        
        # Add statistics
        island_data["island_id"] = idx
        island_data["statistics"] = {
            "abstract_issue_count": len(island_data["abstract_issues"]),
            "abstract_solution_count": len(island_data["abstract_solutions"]),
            "relation_count": len(island_data["relations"]),
            "total_grouped_people": sum(
                len(ai["grouped_people"]) for ai in island_data["abstract_issues"]
            ) + sum(
                len(asol["grouped_people"]) for asol in island_data["abstract_solutions"]
            )
        }
        
        print(f"    ✅ Extracted: {island_data['statistics']['abstract_issue_count']} Issues, "
              f"{island_data['statistics']['abstract_solution_count']} Solutions, "
              f"{island_data['statistics']['total_grouped_people']} GroupedPeople, "
              f"{island_data['statistics']['relation_count']} Relations")
        
        all_islands_data.append(island_data)
    
    driver.close()
    
    # 3. Save results
    OUTPUT_DIR = "/workspace/outputs_sample/islands"
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_filename = f"islands_export_{timestamp}.json"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    
    try:
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(all_islands_data, f, ensure_ascii=False, indent=2)
        print(f"\n💾 Islands data saved to: {output_path}")
    except Exception as e:
        print(f"❌ Failed to save islands data: {e}")
    
    # 4. Show summary
    print(f"\n📊 Phase 3-1 Summary:")
    print(f"  🏝️ Total islands extracted: {len(all_islands_data)}")
    
    total_issues = sum(island["statistics"]["abstract_issue_count"] for island in all_islands_data)
    total_solutions = sum(island["statistics"]["abstract_solution_count"] for island in all_islands_data)
    total_gp = sum(island["statistics"]["total_grouped_people"] for island in all_islands_data)
    total_relations = sum(island["statistics"]["relation_count"] for island in all_islands_data)
    
    print(f"  📋 Total AbstractIssues: {total_issues}")
    print(f"  📋 Total AbstractSolutions: {total_solutions}")
    print(f"  👥 Total GroupedPeople: {total_gp}")
    print(f"  🔗 Total Relations: {total_relations}")
    
    # Details of each island
    if len(all_islands_data) <= 10:  # Show details only when there are 10 or fewer islands
        print(f"\n  🏝️ Island Details:")
        for island in all_islands_data:
            print(f"    Island {island['island_id']}:")
            print(f"      - AbstractIssues: {island['statistics']['abstract_issue_count']}")
            print(f"      - AbstractSolutions: {island['statistics']['abstract_solution_count']}")
            print(f"      - GroupedPeople: {island['statistics']['total_grouped_people']}")
            print(f"      - Relations: {island['statistics']['relation_count']}")
    
    print("\n✅ Phase 3-1 completed")

# Phase 3-2: Associated Personas Generation

In [ ]:
# Run Phase 3-2: Persona creation
print("🎭 PHASE 3-2: Creating personas from island data...")

# 1. Load Phase 3-1 result file (use latest file)
import glob

islands_files = glob.glob("/workspace/outputs_sample/islands/islands_export_*.json")
if not islands_files:
    print("❌ No islands export files found. Please run Phase 3-1 first.")
else:
    # Get latest file
    latest_islands_file = max(islands_files, key=os.path.getctime)
    print(f"📂 Loading islands data from: {latest_islands_file}")
    
    with open(latest_islands_file, "r", encoding="utf-8") as f:
        islands_data = json.load(f)
    
    print(f"Found {len(islands_data)} islands to process")
    
    # 2. Create personas for each island (parallel)
    all_personas = process_personas_parallel(islands_data, max_workers=5)
    
    # 3. Save results
    PERSONA_OUTPUT_DIR = "/workspace/outputs_sample/associated_personas"
    if not os.path.exists(PERSONA_OUTPUT_DIR):
        os.makedirs(PERSONA_OUTPUT_DIR)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    persona_output_filename = f"personas_export_{timestamp}.json"
    persona_output_path = os.path.join(PERSONA_OUTPUT_DIR, persona_output_filename)
    
    try:
        with open(persona_output_path, "w", encoding="utf-8") as f:
            json.dump(all_personas, f, ensure_ascii=False, indent=2)
        print(f"\n💾 Personas data saved to: {persona_output_path}")
    except Exception as e:
        print(f"❌ Failed to save personas data: {e}")
    
    # 4. Show statistics
    success_count = sum(1 for p in all_personas if p.get("status") == "success")
    error_count = sum(1 for p in all_personas if p.get("status") == "error")
    
    # Count scenario pairs (supports new format)
    total_scenario_pairs = sum(
        len(p.get("scenario_pairs", [])) 
        for p in all_personas 
        if p.get("status") == "success"
    )
    
    # Count total number of agent and admin personas
    total_agents = 0
    total_admins = 0
    for p_data in all_personas:
        if p_data.get("status") == "success":
            pairs = p_data.get("scenario_pairs", [])
            total_agents += len(pairs)
            total_admins += len(pairs)
    
    print(f"\n📊 Phase 3-2 Summary:")
    print(f"  🏝️ Islands processed: {len(all_personas)}")
    print(f"    ✅ Successfully created scenario pairs: {success_count} islands")
    print(f"    ❌ Errors: {error_count} islands")
    print(f"\n  🎭 Total scenario pairs created: {total_scenario_pairs}")
    print(f"    👤 Agent personas (Insurance Agency Representative): {total_agents}")
    print(f"    🛠️  Admin personas (Insurance Company System Administrator): {total_admins}")
    
    # Scenario pair details created for each island (first 5 islands only)
    if success_count > 0:
        print(f"\n  🎭 Scenario Pair Details (first {min(5, success_count)} islands):")
        display_count = 0
        for p_data in all_personas:
            if p_data.get("status") == "success" and display_count < 5:
                island_id = p_data.get("island_id")
                pairs = p_data.get("scenario_pairs", [])
                print(f"\n    Island {island_id}: {len(pairs)} scenario pairs")
                for pair in pairs:
                    scenario_id = pair.get("scenario_id", "N/A")
                    context = pair.get("context_summary", "")[:80] + "..." if len(pair.get("context_summary", "")) > 80 else pair.get("context_summary", "")
                    
                    agent = pair.get("agent_persona", {})
                    agent_name = agent.get("basic_profile", {}).get("name", "N/A")
                    agent_role = agent.get("basic_profile", {}).get("role", "N/A")
                    
                    admin = pair.get("admin_persona", {})
                    admin_name = admin.get("basic_profile", {}).get("name", "N/A")
                    admin_role = admin.get("basic_profile", {}).get("role", "N/A")
                    
                    print(f"      - {scenario_id}: {context}")
                    print(f"        Agent: {agent_name} ({agent_role})")
                    print(f"        Admin: {admin_name} ({admin_role})")
                display_count += 1
    
    print("\n✅ Phase 3-2 completed")